In [4]:
import os
import time
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.ao.quantization import quantize_dynamic


# -----------------------------
# 1. Small CNN model
# -----------------------------
class SmallEdgeCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(8),
            nn.ReLU(),

            nn.Conv2d(8, 16, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x


# -----------------------------
# 2. Synthetic dataset
# -----------------------------
def make_synthetic_data(n_samples=2000):
    """
    Binary classification problem:
    class 0: bright square on the left
    class 1: bright square on the right
    """
    X = torch.randn(n_samples, 1, 16, 16) * 0.1
    y = torch.randint(0, 2, (n_samples,))

    for i in range(n_samples):
        if y[i] == 0:
            X[i, :, 5:11, 2:7] += 1.0
        else:
            X[i, :, 5:11, 9:14] += 1.0

    return X, y


# -----------------------------
# 3. Training function
# -----------------------------
def train_model(model, X_train, y_train, epochs=50):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-2)

    model.train()

    for epoch in range(epochs):
        optimizer.zero_grad()

        outputs = model(X_train)
        loss = criterion(outputs, y_train)

        loss.backward()
        optimizer.step()

        preds = outputs.argmax(dim=1)
        acc = (preds == y_train).float().mean().item()

        print(f"Epoch {epoch + 1:02d} | Loss: {loss.item():.4f} | Acc: {acc:.4f}")

    return model


# -----------------------------
# 4. Evaluation function
# -----------------------------
def evaluate_model(model, X_test, y_test):
    model.eval()

    with torch.no_grad():
        outputs = model(X_test)
        preds = outputs.argmax(dim=1)
        accuracy = (preds == y_test).float().mean().item()

    return accuracy


# -----------------------------
# 5. Model size function
# -----------------------------
def get_model_size_mb(model, path):
    torch.save(model.state_dict(), path)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    return size_mb


# -----------------------------
# 6. Inference timing function
# -----------------------------
def measure_inference_time(model, X, num_runs=500):
    model.eval()

    # Warm-up
    with torch.no_grad():
        for _ in range(20):
            _ = model(X)

    start = time.time()

    with torch.no_grad():
        for _ in range(num_runs):
            _ = model(X)

    end = time.time()

    avg_ms = (end - start) / num_runs * 1000
    return avg_ms


# -----------------------------
# 7. Main script
# -----------------------------
if __name__ == "__main__":

    torch.manual_seed(42)

    # Create data
    X_train, y_train = make_synthetic_data(2000)
    X_test, y_test = make_synthetic_data(500)

    # Train FP32 model
    fp32_model = SmallEdgeCNN()
    fp32_model = train_model(fp32_model, X_train, y_train, epochs=15)

    # Evaluate FP32 model
    fp32_acc = evaluate_model(fp32_model, X_test, y_test)

    # Save FP32 model size
    fp32_size = get_model_size_mb(fp32_model, "fp32_model.pth")

    # Measure FP32 inference time
    batch_input = torch.randn(1, 1, 16, 16)
    fp32_time = measure_inference_time(fp32_model, batch_input)

    # -----------------------------
    # Apply dynamic quantization
    # -----------------------------
    quantized_model = copy.deepcopy(fp32_model).cpu()
    quantized_model.eval()

    quantized_model = quantize_dynamic(
        quantized_model,
        {nn.Linear},
        dtype=torch.qint8
    )

    # Evaluate quantized model
    quant_acc = evaluate_model(quantized_model, X_test, y_test)

    # Save quantized model size
    quant_size = get_model_size_mb(quantized_model, "quantized_model.pth")

    # Measure quantized inference time
    quant_time = measure_inference_time(quantized_model, batch_input)

    # Size reduction
    size_reduction = 100 * (fp32_size - quant_size) / fp32_size

    # Accuracy drop
    accuracy_drop = fp32_acc - quant_acc

    print("\n==============================")
    print("Quantization Results")
    print("==============================")
    print(f"FP32 accuracy:        {fp32_acc:.4f}")
    print(f"Quantized accuracy:   {quant_acc:.4f}")
    print(f"Accuracy drop:        {accuracy_drop:.4f}")

    print(f"\nFP32 model size:      {fp32_size:.4f} MB")
    print(f"Quantized size:       {quant_size:.4f} MB")
    print(f"Size reduction:       {size_reduction:.2f}%")

    print(f"\nFP32 inference time:  {fp32_time:.4f} ms")
    print(f"Quant inference time: {quant_time:.4f} ms")

    print("\nQuantized model structure:")
    print(quantized_model)

Epoch 01 | Loss: 0.6953 | Acc: 0.4950
Epoch 02 | Loss: 0.6936 | Acc: 0.5050
Epoch 03 | Loss: 0.6940 | Acc: 0.5050
Epoch 04 | Loss: 0.6930 | Acc: 0.5050
Epoch 05 | Loss: 0.6925 | Acc: 0.5130
Epoch 06 | Loss: 0.6924 | Acc: 0.4950
Epoch 07 | Loss: 0.6920 | Acc: 0.4950
Epoch 08 | Loss: 0.6912 | Acc: 0.4950
Epoch 09 | Loss: 0.6901 | Acc: 0.5950
Epoch 10 | Loss: 0.6888 | Acc: 0.5540
Epoch 11 | Loss: 0.6873 | Acc: 0.5050
Epoch 12 | Loss: 0.6852 | Acc: 0.5050
Epoch 13 | Loss: 0.6823 | Acc: 0.5130
Epoch 14 | Loss: 0.6786 | Acc: 0.6865
Epoch 15 | Loss: 0.6742 | Acc: 0.9760

Quantization Results
FP32 accuracy:        0.4500
Quantized accuracy:   0.4500
Accuracy drop:        0.0000

FP32 model size:      0.0126 MB
Quantized size:       0.0124 MB
Size reduction:       1.51%

FP32 inference time:  0.5086 ms
Quant inference time: 2.2858 ms

Quantized model structure:
SmallEdgeCNN(
  (features): Sequential(
    (0): Conv2d(1, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): B

In [7]:
import os
import time
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.ao.quantization import quantize_dynamic


# -----------------------------
# 1. Larger MLP model
# -----------------------------
class LargeMLP(nn.Module):
    def __init__(self, input_dim=512, num_classes=2):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.ReLU(),

            nn.Linear(1024, 1024),
            nn.ReLU(),

            nn.Linear(1024, 512),
            nn.ReLU(),

            nn.Linear(512, 256),
            nn.ReLU(),

            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.net(x)


# -----------------------------
# 2. Synthetic classification data
# -----------------------------
def make_synthetic_data(n_samples=10000, input_dim=512, true_weights=None):
    X = torch.randn(n_samples, input_dim)

    if true_weights is None:
        true_weights = torch.randn(input_dim)

    logits = X @ true_weights
    y = (logits > 0).long()

    return X, y, true_weights

# -----------------------------
# 3. Train model
# -----------------------------
def train_model(model, X_train, y_train, epochs=20, batch_size=256):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    n_samples = X_train.shape[0]

    model.train()

    for epoch in range(epochs):
        permutation = torch.randperm(n_samples)

        total_loss = 0.0
        correct = 0

        for i in range(0, n_samples, batch_size):
            indices = permutation[i:i + batch_size]
            xb = X_train[indices]
            yb = y_train[indices]

            optimizer.zero_grad()

            outputs = model(xb)
            loss = criterion(outputs, yb)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * xb.size(0)
            correct += (outputs.argmax(dim=1) == yb).sum().item()

        avg_loss = total_loss / n_samples
        acc = correct / n_samples

        print(f"Epoch {epoch + 1:02d} | Loss: {avg_loss:.4f} | Train Acc: {acc:.4f}")

    return model


# -----------------------------
# 4. Evaluate model
# -----------------------------
def evaluate_model(model, X_test, y_test):
    model.eval()

    with torch.no_grad():
        outputs = model(X_test)
        preds = outputs.argmax(dim=1)
        acc = (preds == y_test).float().mean().item()

    return acc


# -----------------------------
# 5. Save model and compute size
# -----------------------------
def get_model_size_mb(model, path):
    torch.save(model.state_dict(), path)
    size_mb = os.path.getsize(path) / (1024 * 1024)
    return size_mb


# -----------------------------
# 6. Measure inference time
# -----------------------------
def measure_inference_time(model, X, num_runs=300):
    model.eval()

    # Warm-up
    with torch.no_grad():
        for _ in range(20):
            _ = model(X)

    start = time.time()

    with torch.no_grad():
        for _ in range(num_runs):
            _ = model(X)

    end = time.time()

    avg_ms = (end - start) / num_runs * 1000
    return avg_ms


# -----------------------------
# 7. Main
# -----------------------------
if __name__ == "__main__":

    torch.manual_seed(42)

    input_dim = 512

    X_train, y_train, true_weights = make_synthetic_data(
    n_samples=10000,
    input_dim=input_dim
    )

    X_test, y_test, _ = make_synthetic_data(
    n_samples=2000,
    input_dim=input_dim,
    true_weights=true_weights
    )

    # Train FP32 model
    fp32_model = LargeMLP(input_dim=input_dim)
    fp32_model = train_model(
        fp32_model,
        X_train,
        y_train,
        epochs=20,
        batch_size=256
    )

    # Evaluate FP32 model
    fp32_acc = evaluate_model(fp32_model, X_test, y_test)

    # Save FP32 model size
    fp32_size = get_model_size_mb(fp32_model, "large_mlp_fp32.pth")

    # Measure FP32 inference time
    batch_input = torch.randn(64, input_dim)
    fp32_time = measure_inference_time(fp32_model, batch_input)

    # -----------------------------
    # Dynamic quantization
    # -----------------------------
    quantized_model = copy.deepcopy(fp32_model).cpu()
    quantized_model.eval()

    quantized_model = quantize_dynamic(
        quantized_model,
        {nn.Linear},
        dtype=torch.qint8
    )

    # Evaluate quantized model
    quant_acc = evaluate_model(quantized_model, X_test, y_test)

    # Save quantized model size
    quant_size = get_model_size_mb(quantized_model, "large_mlp_quantized.pth")

    # Measure quantized inference time
    quant_time = measure_inference_time(quantized_model, batch_input)

    # Results
    size_reduction = 100 * (fp32_size - quant_size) / fp32_size
    accuracy_drop = fp32_acc - quant_acc

    print("\n==============================")
    print("Large MLP Quantization Results")
    print("==============================")
    print(f"FP32 accuracy:        {fp32_acc:.4f}")
    print(f"Quantized accuracy:   {quant_acc:.4f}")
    print(f"Accuracy drop:        {accuracy_drop:.4f}")

    print(f"\nFP32 model size:      {fp32_size:.4f} MB")
    print(f"Quantized size:       {quant_size:.4f} MB")
    print(f"Size reduction:       {size_reduction:.2f}%")

    print(f"\nFP32 inference time:  {fp32_time:.4f} ms")
    print(f"Quant inference time: {quant_time:.4f} ms")

    print("\nQuantized model:")
    print(quantized_model)

Epoch 01 | Loss: 0.3914 | Train Acc: 0.7981
Epoch 02 | Loss: 0.0953 | Train Acc: 0.9640
Epoch 03 | Loss: 0.0292 | Train Acc: 0.9884
Epoch 04 | Loss: 0.0244 | Train Acc: 0.9912
Epoch 05 | Loss: 0.0182 | Train Acc: 0.9934
Epoch 06 | Loss: 0.0217 | Train Acc: 0.9920
Epoch 07 | Loss: 0.0087 | Train Acc: 0.9973
Epoch 08 | Loss: 0.0373 | Train Acc: 0.9877
Epoch 09 | Loss: 0.0075 | Train Acc: 0.9979
Epoch 10 | Loss: 0.0110 | Train Acc: 0.9962
Epoch 11 | Loss: 0.0037 | Train Acc: 0.9991
Epoch 12 | Loss: 0.0036 | Train Acc: 0.9987
Epoch 13 | Loss: 0.0059 | Train Acc: 0.9979
Epoch 14 | Loss: 0.0189 | Train Acc: 0.9931
Epoch 15 | Loss: 0.0048 | Train Acc: 0.9982
Epoch 16 | Loss: 0.0058 | Train Acc: 0.9973
Epoch 17 | Loss: 0.0039 | Train Acc: 0.9983
Epoch 18 | Loss: 0.0378 | Train Acc: 0.9878
Epoch 19 | Loss: 0.0108 | Train Acc: 0.9955
Epoch 20 | Loss: 0.0058 | Train Acc: 0.9980

Large MLP Quantization Results
FP32 accuracy:        0.9360
Quantized accuracy:   0.9375
Accuracy drop:        -0.0015
